# 1D J1J2J3 Inference: LorentzRNN (seed 111-555)

This is part of the work arxiv:2604.24337 [quant-ph] by HL Dao. For the purpose of reproducing the results, the paths to the trained weights have to be adjusted to match the repo structure.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import time
sys.path.append('../../1_hypnqs_torch_lorentz/utility_lorentz')
from j1j2j3_train_loop_lorentz import *

Hypercore Lorentzian module loaded successfully with Geoopt wrappers.
GPU Available:  False


In [2]:
def set_cpu_deterministic(seed):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

In [3]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    # We create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped  
    
def define_load_test_no_tau(wf,  numsamples,path_to_weights, Ee, clipped_e = False):
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    #  Load the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        test_samples_after = wf.sample_no_tau(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = J1J2J3_local_energies(wf, N, J1, J2, J3, Bz, numsamples, test_samples_after, Marshall_sign=True)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2J3_local_energies(wf, N, J1, J2, J3,Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [4]:
N=100
numsamples = 10000
units = 80
syssize = 100
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)
fname='results_LorentzRNN_sc=4.0'

## J2=0.0, J3=0.5

Seeds 333, 444 didn't converge. 

In [5]:
J2_ = 0.0
J3_ = 0.5
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_00_05=-53.9914

In [6]:
spatial_clamp =4.0
seed=111
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed_{seed}/N100_J1=1.0|J2=0.0|J3=0.5_N=100_LorentzRNN_80_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 6,965
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-53.73429870605469-0.0017000000225380063j), var E = 1.2615000009536743
DMRG energy  is -53.9914
Time taken=0.349 hrs


In [7]:
spatial_clamp =4.0
seed=222
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed_{seed}/N100_J1=1.0|J2=0.0|J3=0.5_N=100_LorentzRNN_80_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 6,965
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-53.62950134277344+0.002199999988079071j), var E = 3.0039000511169434
DMRG energy  is -53.9914
Time taken=0.471 hrs


In [8]:
spatial_clamp =4.0
seed=555
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed_{seed}/N100_J1=1.0|J2=0.0|J3=0.5_N=100_LorentzRNN_80_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf , numsamples, h_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 6,965
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-53.93170166015625+0.006200000178068876j), var E = 9.356100082397461
DMRG energy  is -53.9914
Time taken=0.555 hrs


## J2=0.2, J3=0.2

Seeds 222, 333, 555 didnt converge. 

In [5]:
J2_ = 0.2
J3_ = 0.2
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_02_02=-43.5860

In [12]:
spatial_clamp=4.0
seed=111
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units, spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed_{seed}/N100_J1=1.0|J2=0.2|J3=0.2_N=100_LorentzRNN_80_ns=80_MsTrue_s111_checkpoint.pt'

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 6,965
Successfully remapped and loaded weights.
Clipped 276 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.42129898071289+0.0020000000949949026j), var E = 0.20059999823570251
DMRG energy  is -43.586
Time taken=1.423 hrs


In [6]:
spatial_clamp=4.0
seed=444
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units, spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed_{seed}/N100_J1=1.0|J2=0.2|J3=0.2_N=100_LorentzRNN_80_ns=80_MsTrue_s444_checkpoint.pt'

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 6,965
Successfully remapped and loaded weights.
Clipped 263 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.431400299072266-0.0012000000569969416j), var E = 0.33570000529289246
DMRG energy  is -43.586
Time taken=0.577 hrs


## J2=0.2, J3=0.5

Seed 444 didnt converge.

In [7]:
J2_ = 0.2
J3_ = 0.5
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_02_05=-49.6287
spatial_clamp=4.0

In [9]:
seed=111
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=4.0, seed=seed)
h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed_{seed}/N100_J1=1.0|J2=0.2|J3=0.5_N=100_LorentzRNN_80_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 6,965
Successfully remapped and loaded weights.
Clipped 297 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-49.17369842529297-0.00800000037997961j), var E = 1.2972999811172485
DMRG energy  is -49.6287
Time taken=1.048 hrs


In [17]:
seed=222
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=4.0, seed=seed)
h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed_{seed}/N100_J1=1.0|J2=0.2|J3=0.5_N=100_LorentzRNN_80_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 6,965
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-49.54290008544922+0.0013000000035390258j), var E = 1.5928000211715698
DMRG energy  is -49.6287
Time taken=0.803 hrs


In [11]:
seed=333
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=4.0, seed=seed)
h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed_{seed}/N100_J1=1.0|J2=0.2|J3=0.5_N=100_LorentzRNN_80_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 6,965
Successfully remapped and loaded weights.
Clipped 270 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-49.55670166015625-0.0010000000474974513j), var E = 0.6840000152587891
DMRG energy  is -49.6287
Time taken=1.047 hrs


In [12]:
seed=555
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=4.0, seed=seed)
h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed_{seed}/N100_J1=1.0|J2=0.2|J3=0.5_N=100_LorentzRNN_80_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 6,965
Successfully remapped and loaded weights.
Clipped 194 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-49.474998474121094+0.0027000000700354576j), var E = 0.7074000239372253
DMRG energy  is -49.6287
Time taken=0.916 hrs


## J2=0.5, J3=0.2
 
Seeds 222, 333, 444 didn't converge. 

In [13]:
J2_ = 0.5
J3_ = 0.2
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_05_02=-38.5473

In [14]:
seed = 111
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=4.0, seed=seed)
h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed_{seed}/N100_J1=1.0|J2=0.5|J3=0.2_N=100_LorentzRNN_80_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_05_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 6,965
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-38.299400329589844+0.0003000000142492354j), var E = 0.5407000184059143
DMRG energy  is -38.5473
Time taken=0.802 hrs


In [17]:
seed = 555
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=4.0, seed=seed)
h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed_{seed}/N100_J1=1.0|J2=0.5|J3=0.2_N=100_LorentzRNN_80_ns=80_MsTrue_s555_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_05_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 6,965
Successfully remapped and loaded weights.
Clipped 211 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-38.32440185546875-0.0026000000070780516j), var E = 0.17739999294281006
DMRG energy  is -38.5473
Time taken=7.796 hrs
